# CONUS Monthly Runoff/Discharge Maps and Animation
Adapted from the sample NASA_Andes discharge-animation notebook. Side-by-side: CTSM gridded runoff (QRUNOFF) vs. mizuRoute basin runoff (basRunoff) or routed discharge (RoutedRunoff), for a single month or as an animation across months.

`basRunoff` is indexed by `hru`/`basinID` and `RoutedRunoff` by `seg`/`reachID`; neither is guaranteed to follow the mesh file's `hruid` order, so both are matched by ID.

Run from this folder. Paths come from `analysis/config/paths.py`, file reading from `analysis/lib/io.py`; figures go to `<output root>/animation`.

In [ ]:
%matplotlib inline
import os, sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
import cartopy.feature as cfeature

sys.path.insert(0, "../config")   # analysis/config/paths.py
sys.path.insert(0, "..")          # analysis/lib
import paths
from lib import io as lio
from lib.plotting import CONUS_EXTENT, PROJ

CASE = "control"
OUT_DIR = f"{paths.OUTPUT_ROOT}/animation"
os.makedirs(OUT_DIR, exist_ok=True)
land = cfeature.LAND

## Variable metadata (color scale, units, conversion)

In [ ]:
var_meta = {
    'RoutedRunoff': {'model': 'mizuroute', 'name': 'Discharge', 'unit': 'm3/s',
                     'conversion': 1, 'norm': mpl.colors.LogNorm(vmin=0.1, vmax=10000)},
    'basRunoff':    {'model': 'mizuroute', 'name': 'runoff', 'unit': 'mm/day',
                     'conversion': 86400, 'norm': mpl.colors.LogNorm(vmin=0.01, vmax=10)},
    'QRUNOFF':      {'model': 'ctsm', 'name': 'runoff', 'unit': 'mm/day',
                     'conversion': 86400, 'norm': mpl.colors.LogNorm(vmin=0.01, vmax=10)},
}

## Load CTSM monthly (h0a) and mizuRoute monthly output

In [ ]:
files_ctsm = lio.lnd_h0_files(paths.ARCHIVE, CASE)
files_rof = lio.rof_files(paths.ARCHIVE, CASE)
print(f"CTSM monthly files: {len(files_ctsm)}, mizuRoute monthly files: {len(files_rof)}")

ds_ctsm = lio.open_land_hist(files_ctsm, ["QRUNOFF"], parallel=True)
ds_rof, reachID_rof = lio.open_mizuroute(files_rof, ["RoutedRunoff", "basRunoff"], parallel=True)
basinID_rof = xr.open_dataset(files_rof[0])["basinID"].values   # static, from one file

ds_mesh_rof = xr.open_dataset(paths.ROF_MESH)
seg_lon = ds_mesh_rof["centerCoords"].values[:, 0]
seg_lat = ds_mesh_rof["centerCoords"].values[:, 1]
seg_lon_0360 = np.where(seg_lon < 0, seg_lon + 360, seg_lon)

## Match reachID / basinID to mesh coordinates by ID, not position

In [ ]:
mesh_lookup = pd.DataFrame({
    "hruid": ds_mesh_rof["hruid"].values,
    "lon": seg_lon_0360,
    "lat": seg_lat,
}).set_index("hruid")

matched_seg = mesh_lookup.reindex(reachID_rof)
matched_bas = mesh_lookup.reindex(basinID_rof)
print(f"RoutedRunoff unmatched: {matched_seg['lon'].isna().sum()} / {len(reachID_rof)}")
print(f"basRunoff unmatched: {matched_bas['lon'].isna().sum()} / {len(basinID_rof)}")

def get_coords_for(rof_var):
    m = matched_bas if rof_var == "basRunoff" else matched_seg
    return m["lon"].values, m["lat"].values

## Single-month comparison plot (test before animating)

In [ ]:
def plot_month(tindex, rof_var="RoutedRunoff", ctsm_var="QRUNOFF", savepath=False):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True,
                           subplot_kw={"projection": PROJ})
    ds_s = ds_ctsm.isel(time=tindex, drop=True).load()
    ds_r = ds_rof.isel(time=tindex, drop=True).load()
    datetime_str = ds_ctsm["time"].isel(time=tindex).values.item().strftime("%Y-%m")

    ax[0].add_feature(land, facecolor="xkcd:light gray", edgecolor="k", alpha=0.3, lw=0.2, zorder=0)
    field = ds_s[ctsm_var] * var_meta[ctsm_var]["conversion"]
    field.plot.pcolormesh(ax=ax[0], x="lon", y="lat", cmap="YlGnBu",
                          norm=var_meta[ctsm_var]["norm"], add_colorbar=True,
                          cbar_kwargs={"label": f"{var_meta[ctsm_var]['name']} [{var_meta[ctsm_var]['unit']}]",
                                       "orientation": "horizontal"})
    ax[0].set_title(f"{datetime_str} CTSM {var_meta[ctsm_var]['name']}")

    plot_lon, plot_lat = get_coords_for(rof_var)
    ax[1].add_feature(land, facecolor="xkcd:light gray", edgecolor="k", alpha=0.3, lw=0.2, zorder=0)
    values = (ds_r[rof_var] * var_meta[rof_var]["conversion"]).values
    sc = ax[1].scatter(plot_lon, plot_lat, c=values, s=2, cmap="YlGnBu",
                       norm=var_meta[rof_var]["norm"], transform=PROJ)
    plt.colorbar(sc, ax=ax[1], label=f"{var_meta[rof_var]['name']} [{var_meta[rof_var]['unit']}]",
                 orientation="horizontal", extend="both")
    ax[1].set_title(f"{datetime_str} mizuRoute {var_meta[rof_var]['name']}")

    for a in ax:
        a.set_extent(CONUS_EXTENT, crs=PROJ)
        a.set_aspect("equal")
    if savepath:
        fig.savefig(savepath, dpi=120, bbox_inches="tight")
    plt.show()

plot_month(12*10+7, rof_var="basRunoff", savepath=f"{OUT_DIR}/test_month_basrunoff.png")

## Animate across months
Start with a short test range before the full 456-month animation.

In [ ]:
N_MONTHS = 24     # len(ds_ctsm["time"]) for the full run
rof_var, ctsm_var = "basRunoff", "QRUNOFF"
plot_lon, plot_lat = get_coords_for(rof_var)

fig, ax = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True,
                       subplot_kw={"projection": PROJ})
for a in ax:
    a.add_feature(land, facecolor="xkcd:light gray", edgecolor="k", alpha=0.3, lw=0.2, zorder=0)
    a.set_extent(CONUS_EXTENT, crs=PROJ)
    a.set_aspect("equal")

field0 = (ds_ctsm[ctsm_var].isel(time=0) * var_meta[ctsm_var]["conversion"]).load()
mesh0 = ax[0].pcolormesh(field0["lon"], field0["lat"], field0.values,
                         cmap="YlGnBu", norm=var_meta[ctsm_var]["norm"], transform=PROJ)
plt.colorbar(mesh0, ax=ax[0], label=f"{var_meta[ctsm_var]['name']} [{var_meta[ctsm_var]['unit']}]")
values0 = (ds_rof[rof_var].isel(time=0) * var_meta[rof_var]["conversion"]).values
sc0 = ax[1].scatter(plot_lon, plot_lat, c=values0, s=3, cmap="YlGnBu",
                    norm=var_meta[rof_var]["norm"], transform=PROJ)
plt.colorbar(sc0, ax=ax[1], label=f"{var_meta[rof_var]['name']} [{var_meta[rof_var]['unit']}]")
title0, title1 = ax[0].set_title(""), ax[1].set_title("")

def update(tindex):
    datetime_str = ds_ctsm["time"].isel(time=tindex).values.item().strftime("%Y-%m")
    field = (ds_ctsm[ctsm_var].isel(time=tindex) * var_meta[ctsm_var]["conversion"]).load()
    mesh0.set_array(field.values.ravel())
    title0.set_text(f"{datetime_str} CTSM {var_meta[ctsm_var]['name']}")
    values = (ds_rof[rof_var].isel(time=tindex) * var_meta[rof_var]["conversion"]).values
    sc0.set_array(values)
    title1.set_text(f"{datetime_str} mizuRoute {var_meta[rof_var]['name']}")
    return mesh0, sc0, title0, title1

anim = FuncAnimation(fig, update, frames=N_MONTHS, blit=False)
anim.save(f"{OUT_DIR}/discharge_animation_{rof_var}.gif", writer="pillow", fps=2)
print(f"Saved animation to {OUT_DIR}/discharge_animation_{rof_var}.gif")
plt.close(fig)